# 📊 Base Histórica v3 — Pipeline com ELO + Regressão Logística
### Notebook 1 de 2: Coleta · Features · ELO · Modelo · Exportação

**Arquitetura nova (vs v2):**
| Antigo | Novo |
|---|---|
| calcF() com CS%/1ºMarc | ELO por liga |
| Scores 0-100 fixos | Probabilidade real 0-1 por mercado |
| FORTE/MODERADO | EV% como critério de entrada |
| Odd como input do modelo | Odd só no filtro de valor (EV) |

**Outputs:**
- `raw_2526.csv` — ledger bruto incremental (preservado entre rodadas)
- `base_historica_v3.csv` — features N10 + ELO + probabilidades + labels
- `modelo_v3.pkl` — modelos treinados por mercado (usado pelo Notebook 2)
- `elo_ratings.json` — ratings ELO atuais de todos os times (usado pelo Notebook 2)


In [12]:
import requests, pandas as pd, numpy as np
import time, os, json, pickle, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

# ── Configuração ──────────────────────────────────────────────────────────────
API_KEY  = "033f6d5fc7b707b144fbd2c28898c1540eb066ee9efea6c038d740ffedeacb5f"
BASE_URL = "https://api.football-data-api.com"
RAW_CSV  = "raw_2526.csv"
BASE_CSV = "base_historica_v3.csv"
PKL_FILE = "modelo_v3.pkl"
ELO_FILE = "elo_ratings.json"
JANELA   = 10          # só N10 — comprovado melhor que N5
MAX_PAGE = 300
SLEEP    = 0.6
ELO_K    = 32          # velocidade de aprendizado ELO (padrão FIFA)
ELO_HOME = 50          # bônus vantagem de jogar em casa
ELO_BASE = 1500        # rating inicial de todo time novo
EV_MIN   = 0.03        # edge mínimo para considerar entrada (3%)

# ── Features que o modelo usa (sem CS% e 1ºMarcador) ─────────────────────────
# Só variáveis com correlação real confirmada na calibração
FEATURES = [
    "elo_delta",        # ELO casa - ELO fora (+home_adv) — substitui calcF
    "xg_n10_diff",      # diferença de XG médio N10
    "xg_pre_diff",      # diferença de XG pré-jogo (quando disponível)
    "pct_vit_diff",     # diferença de % vitória N10
    "ppg_diff",         # diferença de pontos por jogo
    "gols_marc_diff",   # diferença de gols marcados N10
    "gols_sof_diff",    # diferença de gols sofridos N10
]

# ── Mercados e configurações ──────────────────────────────────────────────────
MERCADOS = [
    # (label, nome, odd_tipica, col_prob, col_ev)
    ("label_home_win",   "Back Casa",    1.90, "prob_bc",      "ev_bc"),
    ("label_away_win",   "Back Fora",    2.50, "prob_bf",      "ev_bf"),
    ("label_btts",       "BTTS",         1.85, "prob_btts",    "ev_btts"),
    ("label_over25",     "Over 2.5",     1.80, "prob_over25",  "ev_over25"),
    ("label_over05ht",   "Over 0.5 HT",  1.40, "prob_o05ht",   "ev_o05ht"),
    ("label_btts_ht",    "BTTS HT",      3.50, "prob_btts_ht", "ev_btts_ht"),
]

# ── Ligas e season_ids 2025/26 ────────────────────────────────────────────────
LIGAS = {
    "Alemanha_1 Bundesliga":        14968,
    "Alemanha_2 2. Bundesliga":     14931,
    "Alemanha_3 3. Liga":           14977,
    "Australia_1 A-League":         16036,
    # "Belgica_1 Pro League":         14937,  # ⚠️ TEMPORADA 2025/26 ENCERRADA — aguardar novo season_id
    "Bulgaria_1 First League":      15056,
    "Croacia_1 HNL":                15053,
    "Escocia_1 Premiership":        15000,
    "Espanha_1 La Liga":            14956,
    "Espanha_2 Segunda Division":   15066,
    "Franca_1 Ligue 1":             14932,
    "Franca_2 Ligue 2":             14954,
    # "Grecia_1 Super League":        15163,  # ⚠️ TEMPORADA 2025/26 ENCERRADA — aguardar novo season_id
    "Holanda_1 Eredivisie":         14936,
    "Holanda_2 Eerste Divisie":     14987,
    "Hungria_1 NB I":               14963,
    "Inglaterra_1 Premier League":  15050,
    "Inglaterra_2 Championship":    14930,
    "Inglaterra_3 League One":      14934,
    "Italia_1 Serie A":             15068,
    "Italia_2 Serie B":             15632,
    "Portugal_1 Primeira Liga":     15115,
    "Rep_Tcheca_1 Czech Liga":      14973,
    "Turquia_1 Super Lig":          14972,
    # Sul-americanas → temporada 2026
    "Brasileiro_A Serie A":         16544,
    "Argentina_1 Liga Profesional": 16571,
    "Chile_1 Primera Division":     16615,
    "Colombia_1 Liga BetPlay":      16614,
    # ── Novas ligas adicionadas ───────────────────────────────────────────
    "Suecia_1 Allsvenskan":         16576,   # começa 05/04/2026
    "Suecia_2 Superettan":          16575,   # começa 04/04/2026
    "Noruega_1 Eliteserien":        16558,   # começa 06/04/2026
    "Noruega_2 First Division":     16560,   # começa 12/04/2026
    "Brasileiro_B Serie B":         16783,   # começa 19/04/2026
    "Brasileiro_C Serie C":         14383,   # season 2025 (2026 não disponível ainda)
}

print(f"✅ Configuração carregada")
print(f"   {len(LIGAS)} ligas | Janela N{JANELA} | ELO K={ELO_K} | EV mínimo={EV_MIN*100:.0f}%")
print(f"   Features do modelo: {FEATURES}")


✅ Configuração carregada
   32 ligas | Janela N10 | ELO K=32 | EV mínimo=3%
   Features do modelo: ['elo_delta', 'xg_n10_diff', 'xg_pre_diff', 'pct_vit_diff', 'ppg_diff', 'gols_marc_diff', 'gols_sof_diff']


In [13]:
# Carrega ledger existente sem consumir cota da API
if os.path.exists(RAW_CSV):
    df_raw = pd.read_csv(RAW_CSV, dtype={"fixture_id": str,
                                          "home_id": str,
                                          "away_id": str})
    df_raw = df_raw.drop_duplicates(subset=["fixture_id"])
    ids_existentes = set(df_raw["fixture_id"])
    print(f"📂 Ledger carregado: {len(df_raw):,} partidas | {df_raw['league'].nunique()} ligas")
    print(f"   Período: {pd.to_datetime(df_raw['date_unix'],unit='s').min().date()} → "
          f"{pd.to_datetime(df_raw['date_unix'],unit='s').max().date()}")
    print(f"   Colunas: {list(df_raw.columns)}")
else:
    df_raw = pd.DataFrame()
    ids_existentes = set()
    print("📂 Nenhum ledger encontrado — coleta do zero")


📂 Ledger carregado: 5,795 partidas | 28 ligas
   Período: 2025-07-18 → 2026-03-26
   Colunas: ['fixture_id', 'league', 'season_id', 'date_unix', 'home_id', 'away_id', 'home_team', 'away_team', 'score_home', 'score_away', 'score_home_ht', 'score_away_ht', 'odds_home', 'odds_draw', 'odds_away', 'odds_btts_y', 'odds_over25', 'odds_over05ht', 'xg_home', 'xg_away', 'xg_home_pre', 'xg_away_pre', 'shots_home', 'shots_away', 'shots_ot_home', 'shots_ot_away', 'corners_home', 'corners_away', 'ppg_home', 'ppg_away', 'marcou_pri_home']


In [14]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2b — Verificação de status das ligas na API
# Detecta automaticamente: temporadas encerradas, breaks, season_ids errados
# Execute antes de coletar para evitar coletas desnecessárias
# ═══════════════════════════════════════════════════════════
import datetime, time

agora      = int(datetime.datetime.now().timestamp())
limite_7d  = agora + 7  * 86400
limite_30d = agora + 30 * 86400

print('Verificando status de cada liga na API...')
print(f'Data atual: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')

status_ligas = []
for nome, sid in LIGAS.items():
    try:
        resp = requests.get(
            f'{BASE_URL}/league-matches',
            params={'key': API_KEY, 'season_id': sid, 'max_per_page': 300, 'page': 1},
            timeout=15
        )
        if resp.status_code == 429:
            time.sleep(60)
            resp = requests.get(f'{BASE_URL}/league-matches',
                params={'key': API_KEY, 'season_id': sid, 'max_per_page': 300, 'page': 1}, timeout=15)
        data    = resp.json()
        matches = data.get('data', [])
        total   = data.get('pager', {}).get('total_results', 0)

        completos   = [m for m in matches if m.get('status') == 'complete']
        incompletos = [m for m in matches if m.get('status') != 'complete']
        na_7d       = [m for m in incompletos if agora <= (m.get('date_unix',0) or 0) <= limite_7d]
        no_30d      = [m for m in incompletos if agora <= (m.get('date_unix',0) or 0) <= limite_30d]

        ultimo_dt = ''
        if completos:
            ult = max(completos, key=lambda x: x.get('date_unix',0))
            ultimo_dt = datetime.datetime.fromtimestamp(ult['date_unix']).strftime('%Y-%m-%d')

        proximo_dt = ''
        if no_30d:
            prox = min(no_30d, key=lambda x: x.get('date_unix',0))
            proximo_dt = datetime.datetime.fromtimestamp(prox['date_unix']).strftime('%Y-%m-%d')

        # Classificar status
        if len(incompletos) == 0:
            st = '🔴 TEMPORADA ENCERRADA'
        elif len(na_7d) > 0:
            st = f'✅ ATIVA ({len(na_7d)} jogos esta semana)'
        elif len(no_30d) > 0:
            st = f'⏸️  BREAK (próximo: {proximo_dt})'
        else:
            st = '⚠️  SEM JOGOS NOS PRÓXIMOS 30 DIAS'

        status_ligas.append({
            'liga': nome, 'season_id': sid, 'total': total,
            'n_7d': len(na_7d), 'ultimo': ultimo_dt, 'proximo': proximo_dt, 'status': st
        })
        time.sleep(0.4)
    except Exception as e:
        status_ligas.append({'liga': nome, 'season_id': sid, 'total': 0,
                             'n_7d': 0, 'ultimo': '?', 'proximo': '?', 'status': f'❌ ERRO: {e}'})

# Exibir relatório
print(f'{"LIGA":<45} {"N_7D":>5} {"ÚLTIMO":>12} {"PRÓXIMO":>12}  STATUS')
print('-' * 100)
for r in status_ligas:
    print(f'{r["liga"]:<45} {r["n_7d"]:>5} {r["ultimo"]:>12} {r["proximo"]:>12}  {r["status"]}')

# Alertas críticos
encerradas = [r for r in status_ligas if 'ENCERRADA' in r['status']]
if encerradas:
    print(f'\n⚠️  ATENÇÃO — {len(encerradas)} liga(s) com TEMPORADA ENCERRADA:')
    for r in encerradas:
        print(f'   ✗ {r["liga"]} (season_id={r["season_id"]}) — Remover de LIGAS ou atualizar season_id')
else:
    print('\n✅ Nenhuma liga com temporada encerrada detectada')


Verificando status de cada liga na API...
Data atual: 2026-03-27 18:01

LIGA                                           N_7D       ÚLTIMO      PRÓXIMO  STATUS
----------------------------------------------------------------------------------------------------
Alemanha_1 Bundesliga                             0   2026-03-22   2026-04-04  ⏸️  BREAK (próximo: 2026-04-04)
Alemanha_2 2. Bundesliga                          0   2026-03-22   2026-04-04  ⏸️  BREAK (próximo: 2026-04-04)
Alemanha_3 3. Liga                                0   2026-03-22   2026-04-04  ⏸️  BREAK (próximo: 2026-04-04)
Australia_1 A-League                              3   2026-03-22   2026-04-02  ✅ ATIVA (3 jogos esta semana)
Bulgaria_1 First League                           1   2026-03-22   2026-04-03  ✅ ATIVA (1 jogos esta semana)
Croacia_1 HNL                                     1   2026-03-22   2026-04-03  ✅ ATIVA (1 jogos esta semana)
Escocia_1 Premiership                             0   2026-03-22   2026-04-04  ⏸️

In [15]:
def coletar_liga(liga_nome, season_id, ids_existentes):
    """
    Coleta partidas encerradas de uma liga.
    Campos corretos da API FootyStats (confirmados):
      homeID/awayID | ht_goals_team_a/b | team_a/b_xg | odds_ft_1/x/2
    """
    novos, page = [], 1
    while True:
        try:
            resp = requests.get(
                f"{BASE_URL}/league-matches",
                params={"key": API_KEY, "season_id": season_id,
                        "max_per_page": MAX_PAGE, "page": page},
                timeout=20
            )
            if resp.status_code == 429:
                print("  ⚠️  Rate limit — aguardando 60s...")
                time.sleep(60); continue
            resp.raise_for_status()
            data = resp.json()
        except Exception as e:
            print(f"  ❌ {liga_nome} p{page}: {e}"); break

        matches = data.get("data", [])
        if not matches: break

        for m in matches:
            fid = str(m.get("id", ""))
            if fid in ids_existentes: continue
            if m.get("status", "") != "complete": continue

            # IDs — campo correto é homeID/awayID (CamelCase)
            hid = m.get("homeID")
            aid = m.get("awayID")
            hid = str(int(hid)) if hid is not None else f"{fid}_h"
            aid = str(int(aid)) if aid is not None else f"{fid}_a"

            # Gols HT — campos separados
            ht_h = int(m.get("ht_goals_team_a", 0) or 0)
            ht_a = int(m.get("ht_goals_team_b", 0) or 0)

            # 1º marcador via gols HT (proxy)
            marcou_pri_home = 1 if ht_h > 0 else 0

            novos.append({
                "fixture_id":      fid,
                "league":          liga_nome,
                "season_id":       season_id,
                "date_unix":       m.get("date_unix", 0),
                "home_id":         hid,
                "away_id":         aid,
                "home_team":       m.get("home_name", ""),
                "away_team":       m.get("away_name", ""),
                # Placar
                "score_home":      int(m.get("homeGoalCount", 0) or 0),
                "score_away":      int(m.get("awayGoalCount", 0) or 0),
                "score_home_ht":   ht_h,
                "score_away_ht":   ht_a,
                # Odds de fechamento (partidas encerradas)
                "odds_home":       m.get("odds_ft_1"),
                "odds_draw":       m.get("odds_ft_x"),
                "odds_away":       m.get("odds_ft_2"),
                "odds_btts_y":     m.get("odds_btts_yes"),
                "odds_over25":     m.get("odds_ft_over25"),
                "odds_over05ht":   m.get("odds_1st_half_over05"),
                # Stats pós-jogo (para features de qualidade)
                "xg_home":         m.get("team_a_xg"),
                "xg_away":         m.get("team_b_xg"),
                "xg_home_pre":     m.get("team_a_xg_prematch"),  # XG pré-jogo
                "xg_away_pre":     m.get("team_b_xg_prematch"),
                "shots_home":      m.get("team_a_shots"),
                "shots_away":      m.get("team_b_shots"),
                "shots_ot_home":   m.get("team_a_shotsOnTarget"),
                "shots_ot_away":   m.get("team_b_shotsOnTarget"),
                "corners_home":    m.get("team_a_corners"),
                "corners_away":    m.get("team_b_corners"),
                "ppg_home":        m.get("pre_match_home_ppg"),
                "ppg_away":        m.get("pre_match_away_ppg"),
                "marcou_pri_home": marcou_pri_home,
            })

        pager = data.get("pager", {})
        if page >= pager.get("total_pages", 1): break
        page += 1
        time.sleep(SLEEP)
    return novos

# ── Executar coleta ───────────────────────────────────────────────────────────
todos_novos = []
for liga_nome, season_id in LIGAS.items():
    print(f"📡 {liga_nome} (id={season_id})")
    novos = coletar_liga(liga_nome, season_id, ids_existentes)
    todos_novos.extend(novos)
    print(f"   ✅ {len(novos)} novas")
    time.sleep(SLEEP)

if todos_novos:
    df_novos = pd.DataFrame(todos_novos)
    df_raw   = pd.concat([df_raw, df_novos], ignore_index=True) if len(df_raw) else df_novos
    df_raw   = df_raw.drop_duplicates(subset=["fixture_id"])
    df_raw["home_id"] = df_raw["home_id"].astype(str)
    df_raw["away_id"] = df_raw["away_id"].astype(str)
    df_raw.to_csv(RAW_CSV, index=False)
    print(f"\n💾 Ledger salvo: {len(df_raw):,} partidas | {len(todos_novos)} novas")
else:
    print("\nℹ️  Nenhuma partida nova encontrada")

display(df_raw.tail(3))


📡 Alemanha_1 Bundesliga (id=14968)
   ✅ 0 novas
📡 Alemanha_2 2. Bundesliga (id=14931)
   ✅ 0 novas
📡 Alemanha_3 3. Liga (id=14977)
   ✅ 0 novas
📡 Australia_1 A-League (id=16036)
   ✅ 0 novas
📡 Bulgaria_1 First League (id=15056)
   ✅ 0 novas
📡 Croacia_1 HNL (id=15053)
   ✅ 0 novas
📡 Escocia_1 Premiership (id=15000)
   ✅ 0 novas
📡 Espanha_1 La Liga (id=14956)
   ✅ 0 novas
📡 Espanha_2 Segunda Division (id=15066)
   ✅ 0 novas
📡 Franca_1 Ligue 1 (id=14932)
   ✅ 0 novas
📡 Franca_2 Ligue 2 (id=14954)
   ✅ 0 novas
📡 Holanda_1 Eredivisie (id=14936)
   ✅ 0 novas
📡 Holanda_2 Eerste Divisie (id=14987)
   ✅ 0 novas
📡 Hungria_1 NB I (id=14963)
   ✅ 0 novas
📡 Inglaterra_1 Premier League (id=15050)
   ✅ 0 novas
📡 Inglaterra_2 Championship (id=14930)
   ✅ 0 novas
📡 Inglaterra_3 League One (id=14934)
   ✅ 0 novas
📡 Italia_1 Serie A (id=15068)
   ✅ 0 novas
📡 Italia_2 Serie B (id=15632)
   ✅ 0 novas
📡 Portugal_1 Primeira Liga (id=15115)
   ✅ 0 novas
📡 Rep_Tcheca_1 Czech Liga (id=14973)
   ✅ 0 novas
📡 Turq

,fixture_id,league,season_id,date_unix,home_id,away_id,home_team,away_team,score_home,score_away,...,xg_away_pre,shots_home,shots_away,shots_ot_home,shots_ot_away,corners_home,corners_away,ppg_home,ppg_away,marcou_pri_home
6024,8336510,Brasileiro_C Serie C,14383,1761422400,628,1055,Ponte Preta,Londrina,2,0,...,1.55,15,13,3,5,6,6,1.92,1.58,0
6025,8336511,Brasileiro_C Serie C,14383,1760817600,1055,628,Londrina,Ponte Preta,0,0,...,1.39,14,5,1,0,5,3,1.60,1.96,0
6026,8337997,Brasileiro_C Serie C,14383,1757197800,2677,1055,São Bernardo,Londrina,2,2,...,1.57,0,0,7,3,15,3,1.58,1.58,0


In [16]:
# ── Features N10 anti-leakage ─────────────────────────────────────────────────
# Perspectiva dupla: cada jogo gera features para home e away separadamente
# shift(1) garante que o jogo N não usa o próprio resultado como feature

def preparar_perspectiva(df, lado):
    d = df.copy()
    total = d["score_home"] + d["score_away"]
    if lado == "home":
        d["gols_marc"]  = d["score_home"]
        d["gols_sof"]   = d["score_away"]
        d["gols_ht"]    = d["score_home_ht"]
        d["vitoria"]    = (d["score_home"] > d["score_away"]).astype(int)
        d["clean_sh"]   = (d["score_away"] == 0).astype(int)
        d["marcou_pri"] = d["marcou_pri_home"].fillna(0).astype(int)
        d["xg"]         = pd.to_numeric(d.get("xg_home", 0), errors="coerce").fillna(0)
        d["shots"]      = pd.to_numeric(d.get("shots_home", 0), errors="coerce").fillna(0)
        d["corners"]    = pd.to_numeric(d.get("corners_home", 0), errors="coerce").fillna(0)
        gcol            = "home_id"
    else:
        d["gols_marc"]  = d["score_away"]
        d["gols_sof"]   = d["score_home"]
        d["gols_ht"]    = d["score_away_ht"]
        d["vitoria"]    = (d["score_away"] > d["score_home"]).astype(int)
        d["clean_sh"]   = (d["score_home"] == 0).astype(int)
        d["marcou_pri"] = (1 - d["marcou_pri_home"].fillna(0)).astype(int)
        d["xg"]         = pd.to_numeric(d.get("xg_away", 0), errors="coerce").fillna(0)
        d["shots"]      = pd.to_numeric(d.get("shots_away", 0), errors="coerce").fillna(0)
        d["corners"]    = pd.to_numeric(d.get("corners_away", 0), errors="coerce").fillna(0)
        gcol            = "away_id"
    d["btts"]     = ((d["score_home"]>0) & (d["score_away"]>0)).astype(int)
    d["over25"]   = (total > 2.5).astype(int)
    d["over05ht"] = ((d["score_home_ht"]+d["score_away_ht"]) > 0.5).astype(int)
    d["btts_ht"]  = ((d["score_home_ht"]>0) & (d["score_away_ht"]>0)).astype(int)
    return d, gcol

def features_n10(grp, prefixo):
    n = JANELA
    p = f"{prefixo}_n{n}_"
    def roll(col):
        return grp[col].shift(1).rolling(n, min_periods=1).mean()
    return pd.DataFrame({
        f"{p}jogos":        grp["gols_marc"].shift(1).rolling(n, min_periods=1).count(),
        f"{p}gols_marc":    roll("gols_marc"),
        f"{p}gols_sof":     roll("gols_sof"),
        f"{p}gols_ht":      roll("gols_ht"),
        f"{p}pct_vitoria":  roll("vitoria"),
        f"{p}pct_btts":     roll("btts"),
        f"{p}pct_over25":   roll("over25"),
        f"{p}pct_over05ht": roll("over05ht"),
        f"{p}pct_btts_ht":  roll("btts_ht"),
        f"{p}pct_cs":       roll("clean_sh"),
        f"{p}avg_xg":       roll("xg"),
        f"{p}avg_shots":    roll("shots"),
        f"{p}avg_corners":  roll("corners"),
    }, index=grp.index)

def build_features(df_raw):
    df = df_raw.copy()
    df["date_unix"]     = pd.to_numeric(df["date_unix"], errors="coerce")
    df["score_home"]    = pd.to_numeric(df["score_home"], errors="coerce").fillna(0).astype(int)
    df["score_away"]    = pd.to_numeric(df["score_away"], errors="coerce").fillna(0).astype(int)
    df["score_home_ht"] = pd.to_numeric(df["score_home_ht"], errors="coerce").fillna(0).astype(int)
    df["score_away_ht"] = pd.to_numeric(df["score_away_ht"], errors="coerce").fillna(0).astype(int)
    df = df.sort_values(["league","date_unix"]).reset_index(drop=True)

    print(f"📊 {len(df):,} partidas | {df['league'].nunique()} ligas")

    all_feats = []
    for lado, pref in [("home","home"),("away","away")]:
        d, gcol = preparar_perspectiva(df, lado)
        parts = []
        for _, grp in d.groupby(["league", gcol], sort=False):
            parts.append(features_n10(grp, pref))
        if not parts:
            print(f"  ⚠️  Sem grupos para {lado}"); continue
        df_feat = pd.concat(parts).sort_index().reindex(df.index)
        all_feats.append(df_feat)
        nan_pct = df_feat.isna().mean().mean()*100
        print(f"  {pref} N{JANELA}: {df_feat.shape[1]} colunas | NaN={nan_pct:.1f}% (primeiros jogos de times novos)")

    result = pd.concat([df] + all_feats, axis=1)
    return result.loc[:, ~result.columns.duplicated()]

df_feat = build_features(df_raw)
print(f"\n✅ Features: {df_feat.shape[0]:,} × {df_feat.shape[1]}")
display(df_feat[[c for c in df_feat.columns if "n10" in c]].tail(50))


📊 6,027 partidas | 31 ligas
  home N10: 13 colunas | NaN=8.3% (primeiros jogos de times novos)
  away N10: 13 colunas | NaN=8.3% (primeiros jogos de times novos)

✅ Features: 6,027 × 57


,home_n10_jogos,home_n10_gols_marc,home_n10_gols_sof,home_n10_gols_ht,home_n10_pct_vitoria,home_n10_pct_btts,home_n10_pct_over25,home_n10_pct_over05ht,home_n10_pct_btts_ht,home_n10_pct_cs,...,away_n10_gols_ht,away_n10_pct_vitoria,away_n10_pct_btts,away_n10_pct_over25,away_n10_pct_over05ht,away_n10_pct_btts_ht,away_n10_pct_cs,away_n10_avg_xg,away_n10_avg_shots,away_n10_avg_corners
5977,10.0,1.700000,1.400000,0.700000,0.500000,0.700000,0.700000,0.900000,0.200000,0.200000,...,0.700000,0.100000,0.800000,0.500000,0.800000,0.500000,0.100000,1.202000,11.400000,3.500000
5978,10.0,0.900000,0.900000,0.600000,0.300000,0.300000,0.300000,0.600000,0.100000,0.600000,...,0.700000,0.200000,0.800000,0.600000,0.900000,0.300000,0.000000,1.393000,12.700000,4.700000
5979,10.0,1.900000,0.800000,0.700000,0.600000,0.600000,0.400000,0.800000,0.200000,0.400000,...,1.200000,0.700000,0.600000,0.600000,0.800000,0.200000,0.400000,1.705000,15.300000,5.700000
5980,10.0,1.000000,0.700000,0.600000,0.500000,0.400000,0.200000,0.600000,0.300000,0.500000,...,0.700000,0.300000,0.600000,0.600000,0.700000,0.200000,0.300000,1.191000,11.300000,2.900000
5981,10.0,1.400000,0.500000,0.800000,0.600000,0.400000,0.400000,0.600000,0.100000,0.600000,...,0.000000,0.100000,0.600000,0.400000,0.400000,0.000000,0.100000,1.251000,11.000000,3.600000
5982,9.0,1.777778,1.333333,0.777778,0.333333,0.777778,0.555556,0.666667,0.333333,0.222222,...,1.300000,0.400000,0.500000,0.500000,1.000000,0.200000,0.300000,1.639000,14.700000,4.500000
5983,9.0,0.666667,1.333333,0.111111,0.000000,0.555556,0.222222,0.555556,0.111111,0.222222,...,0.300000,0.100000,0.600000,0.700000,0.800000,0.200000,0.000000,1.185000,10.900000,3.900000
5984,10.0,1.000000,1.400000,0.400000,0.300000,0.400000,0.500000,0.500000,0.100000,0.400000,...,0.111111,0.222222,0.444444,0.333333,0.222222,0.000000,0.222222,1.083333,9.222222,4.555556
5985,10.0,0.900000,1.300000,0.200000,0.200000,0.500000,0.400000,0.500000,0.000000,0.200000,...,0.400000,0.100000,0.600000,0.500000,0.700000,0.200000,0.100000,0.954000,8.900000,3.400000
5986,10.0,1.100000,1.000000,0.700000,0.400000,0.400000,0.400000,0.700000,0.100000,0.500000,...,0.600000,0.600000,0.400000,0.600000,0.700000,0.000000,0.500000,1.653000,14.700000,5.800000


In [17]:
# ── Motor ELO por liga ────────────────────────────────────────────────────────
# 
# Conceito:
#   Cada time começa em 1500. Após cada jogo:
#     - Vencedor sobe, perdedor desce
#     - A variação depende da surpresa: vencer favorito = sobe muito
#     - K=32 (padrão FIFA), HOME_ADV=50 (vantagem de jogar em casa)
#
# Anti-leakage: salva o ELO ANTES de atualizar com o resultado do jogo
# Isso garante que o modelo usa só informação disponível pré-jogo

elo_ratings = {}  # chave: "league|team_id"

def get_elo(league, tid):
    return elo_ratings.get(f"{league}|{tid}", ELO_BASE)

def set_elo(league, tid, val):
    elo_ratings[f"{league}|{tid}"] = val

elo_h_list, elo_a_list, elo_delta_list = [], [], []

df_sorted = df_feat.sort_values(["league","date_unix"]).reset_index(drop=True)

for _, row in df_sorted.iterrows():
    liga = row["league"]
    hid  = str(row["home_id"])
    aid  = str(row["away_id"])

    elo_h = get_elo(liga, hid)
    elo_a = get_elo(liga, aid)

    # Salva PRÉ-JOGO (anti-leakage)
    elo_h_list.append(elo_h)
    elo_a_list.append(elo_a)
    elo_delta_list.append(elo_h + ELO_HOME - elo_a)  # positivo = casa favorita

    # Resultado real
    sh, sa = int(row["score_home"]), int(row["score_away"])
    if sh > sa:   rh, ra = 1.0, 0.0
    elif sh < sa: rh, ra = 0.0, 1.0
    else:         rh, ra = 0.5, 0.5

    # Probabilidade esperada pelo ELO (com vantagem casa)
    exp_h = 1 / (1 + 10 ** ((elo_a - (elo_h + ELO_HOME)) / 400))
    exp_a = 1 - exp_h

    # Atualiza ratings PÓS-JOGO
    set_elo(liga, hid, elo_h + ELO_K * (rh - exp_h))
    set_elo(liga, aid, elo_a + ELO_K * (ra - exp_a))

df_sorted["elo_home"]  = elo_h_list
df_sorted["elo_away"]  = elo_a_list
df_sorted["elo_delta"] = elo_delta_list

# Diagnóstico
print(f"✅ ELO calculado")
print(f"   Range de ratings: {min(elo_ratings.values()):.0f} → {max(elo_ratings.values()):.0f}")
print(f"   Times trackados: {len(elo_ratings)}")

# Top 10 por ELO atual
top = sorted(elo_ratings.items(), key=lambda x:x[1], reverse=True)[:10]
print("\n   🏆 Top 10 times por ELO atual:")
for k,v in top:
    liga_n = k.split("|")[0].replace("_1 ","_").replace("_2 ","_")[:25]
    print(f"     {liga_n:<28} ELO={v:.0f}")

# Salva ELO para uso no Notebook 2
with open(ELO_FILE, "w") as f:
    json.dump(elo_ratings, f)
print(f"\n💾 {ELO_FILE} salvo ({len(elo_ratings)} times)")

# Reassign df_feat com ELO
df_feat = df_sorted.copy()


✅ ELO calculado
   Range de ratings: 1303 → 1712
   Times trackados: 550

   🏆 Top 10 times por ELO atual:
     Portugal_Primeira Liga       ELO=1712
     Rep_Tcheca_Czech Liga        ELO=1696
     Portugal_Primeira Liga       ELO=1694
     Alemanha_Bundesliga          ELO=1694
     Portugal_Primeira Liga       ELO=1679
     Espanha_La Liga              ELO=1674
     Holanda_Eerste Divisie       ELO=1671
     Croacia_HNL                  ELO=1669
     Turquia_Super Lig            ELO=1659
     Bulgaria_First League        ELO=1659

💾 elo_ratings.json salvo (550 times)


In [18]:
# ── Labels reais ──────────────────────────────────────────────────────────────
df_feat["label_home_win"]    = (df_feat["score_home"] > df_feat["score_away"]).astype(int)
df_feat["label_away_win"]    = (df_feat["score_away"] > df_feat["score_home"]).astype(int)
df_feat["label_draw"]        = (df_feat["score_home"] == df_feat["score_away"]).astype(int)
df_feat["label_btts"]        = ((df_feat["score_home"]>0)&(df_feat["score_away"]>0)).astype(int)
df_feat["label_over25"]      = ((df_feat["score_home"]+df_feat["score_away"])>2.5).astype(int)
df_feat["label_over05ht"]    = ((df_feat["score_home_ht"]+df_feat["score_away_ht"])>0.5).astype(int)
df_feat["label_btts_ht"]     = ((df_feat["score_home_ht"]>0)&(df_feat["score_away_ht"]>0)).astype(int)
df_feat["label_home_win_ht"] = (df_feat["score_home_ht"] > df_feat["score_away_ht"]).astype(int)
df_feat["label_away_win_ht"] = (df_feat["score_away_ht"] > df_feat["score_home_ht"]).astype(int)

print("✅ Labels gerados")
taxa = df_feat[[c for c in df_feat.columns if c.startswith("label_")]].mean().round(3)
print("\nTaxa base de cada evento:")
display(taxa.to_frame("taxa_real"))

# Diagnóstico ELO — correlação com resultado real
corr_elo = df_feat[["elo_delta","label_home_win"]].corr(method="spearman").iloc[0,1]
print(f"\n📊 Correlação elo_delta × label_home_win: {corr_elo:.3f}")
print(f"   (sistema antigo calcF era ~0.05)")


✅ Labels gerados

Taxa base de cada evento:


,taxa_real
label_home_win,0.432
label_away_win,0.302
label_draw,0.265
label_btts,0.535
label_over25,0.514
label_over05ht,0.702
label_btts_ht,0.195
label_home_win_ht,0.333
label_away_win_ht,0.253



📊 Correlação elo_delta × label_home_win: 0.227
   (sistema antigo calcF era ~0.05)


In [19]:
# ── Construir matrix de features para o modelo ───────────────────────────────
#
# Nota sobre xg_pre_diff:
#   Na base histórica = diferença de XG pré-jogo (disponível da API)
#   Nos jogos futuros (Notebook 2) = também disponível da API pré-jogo
#   Quando ausente = 0 (neutro)

df_feat["xg_n10_diff"]    = df_feat["home_n10_avg_xg"].fillna(0)     - df_feat["away_n10_avg_xg"].fillna(0)
df_feat["xg_pre_diff"]    = pd.to_numeric(df_feat["xg_home_pre"], errors="coerce").fillna(0) -                              pd.to_numeric(df_feat["xg_away_pre"], errors="coerce").fillna(0)
df_feat["pct_vit_diff"]   = df_feat["home_n10_pct_vitoria"].fillna(0) - df_feat["away_n10_pct_vitoria"].fillna(0)
df_feat["ppg_diff"]       = pd.to_numeric(df_feat["ppg_home"], errors="coerce").fillna(0) -                              pd.to_numeric(df_feat["ppg_away"], errors="coerce").fillna(0)
df_feat["gols_marc_diff"] = df_feat["home_n10_gols_marc"].fillna(0)  - df_feat["away_n10_gols_marc"].fillna(0)
df_feat["gols_sof_diff"]  = df_feat["home_n10_gols_sof"].fillna(0)   - df_feat["away_n10_gols_sof"].fillna(0)

# Verificar completude das features
mask_completo = df_feat[FEATURES].notna().all(axis=1)
print(f"✅ Features construídas")
print(f"   Jogos com todas as features: {mask_completo.sum():,} ({mask_completo.mean()*100:.1f}%)")
print(f"   Jogos incompletos (primeiras rodadas): {(~mask_completo).sum():,}")
print()

# Correlações rápidas (sanidade)
print("📊 Correlação Spearman de cada feature com label_home_win:")
for feat in FEATURES:
    if feat in df_feat.columns:
        r = df_feat.loc[mask_completo, [feat,"label_home_win"]].corr(method="spearman").iloc[0,1]
        bar = "█" * int(abs(r)*40)
        sinal = "+" if r > 0 else "-"
        print(f"  {feat:<22} {sinal}{abs(r):.3f}  {bar}")


✅ Features construídas
   Jogos com todas as features: 6,027 (100.0%)
   Jogos incompletos (primeiras rodadas): 0

📊 Correlação Spearman de cada feature com label_home_win:
  elo_delta              +0.227  █████████
  xg_n10_diff            +0.206  ████████
  xg_pre_diff            +0.228  █████████
  pct_vit_diff           +0.148  █████
  ppg_diff               +0.174  ██████
  gols_marc_diff         +0.150  █████
  gols_sof_diff          -0.114  ████


In [20]:
# ── Treinar Regressão Logística L1 por mercado ───────────────────────────────
#
# L1 (Lasso): penaliza coeficientes → zera variáveis inúteis automaticamente
# C=1.0: regularização padrão (menor C = mais regularização)
# cross_val AUC: mede poder preditivo real (5 folds)
#
# Saída por jogo: probabilidade real 0-1 (não score 0-100)

X_full = df_feat.loc[mask_completo, FEATURES].values
modelos_salvos = {}

print("="*65)
print("TREINAMENTO — Regressão Logística L1 por mercado")
print("="*65)

for label, nome, odd_tipica, col_prob, col_ev in MERCADOS:
    y = df_feat.loc[mask_completo, label].values

    # Normalizar features (essencial para L1)
    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X_full)

    # Treinar
    model = LogisticRegression(
        penalty="l1", solver="liblinear",
        C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_sc, y)

    # Avaliar com cross-validation
    auc_cv = cross_val_score(model, X_sc, y, cv=5, scoring="roc_auc").mean()

    # Probabilidades e EV
    probs = model.predict_proba(X_sc)[:,1]
    be    = 1 / odd_tipica
    ev    = probs * odd_tipica - 1

    # Jogos com EV > mínimo
    mask_ev  = ev > EV_MIN
    n_ev     = mask_ev.sum()
    acerto_ev= y[mask_ev].mean() if n_ev > 0 else 0
    roi_ev   = acerto_ev * odd_tipica - 1

    # Coeficientes (quais variáveis sobreviveram)
    coefs = dict(zip(FEATURES, model.coef_[0]))
    ativos = [(k,v) for k,v in coefs.items() if abs(v)>0.001]
    ativos.sort(key=lambda x: abs(x[1]), reverse=True)

    print(f"\n{'─'*65}")
    print(f"📊 {nome}")
    print(f"   AUC cross-val: {auc_cv:.3f}  {'✅' if auc_cv>0.60 else '⚠️' if auc_cv>0.55 else '❌'}")
    print(f"   Base rate: {y.mean()*100:.1f}% | Break-even: {be*100:.0f}%")
    print(f"   Jogos com EV>{EV_MIN*100:.0f}%: {n_ev} ({n_ev/len(y)*100:.1f}%)")
    print(f"   Acerto nesses jogos: {acerto_ev*100:.1f}% | ROI: {roi_ev*100:+.1f}%")
    print(f"   Variáveis ativas após L1:")
    for feat, coef in ativos:
        sinal = "▲" if coef>0 else "▼"
        print(f"     {sinal} {feat:<22} {coef:+.4f}")

    # Salvar probabilidades e EV no dataframe
    df_feat.loc[mask_completo, col_prob] = probs
    df_feat.loc[mask_completo, col_ev]   = ev

    # Guardar modelo para serialização
    modelos_salvos[label] = {
        "model": model,
        "scaler": scaler,
        "nome": nome,
        "odd_tipica": odd_tipica,
        "col_prob": col_prob,
        "col_ev": col_ev,
        "auc": auc_cv,
        "features": FEATURES,
    }

print(f"\n{'='*65}")
print("✅ Todos os modelos treinados")


TREINAMENTO — Regressão Logística L1 por mercado

─────────────────────────────────────────────────────────────────
📊 Back Casa
   AUC cross-val: 0.650  ✅
   Base rate: 43.2% | Break-even: 53%
   Jogos com EV>3%: 1225 (20.3%)
   Acerto nesses jogos: 62.7% | ROI: +19.1%
   Variáveis ativas após L1:
     ▲ elo_delta              +0.3763
     ▲ xg_pre_diff            +0.2744
     ▼ pct_vit_diff           -0.0965
     ▲ xg_n10_diff            +0.0635
     ▼ gols_sof_diff          -0.0547
     ▲ ppg_diff               +0.0306
     ▲ gols_marc_diff         +0.0237

─────────────────────────────────────────────────────────────────
📊 Back Fora
   AUC cross-val: 0.646  ✅
   Base rate: 30.2% | Break-even: 40%
   Jogos com EV>3%: 977 (16.2%)
   Acerto nesses jogos: 49.3% | ROI: +23.3%
   Variáveis ativas após L1:
     ▼ elo_delta              -0.4233
     ▼ xg_pre_diff            -0.2180
     ▼ xg_n10_diff            -0.1327
     ▲ pct_vit_diff           +0.0778
     ▲ gols_marc_diff         +0.0

In [21]:
# ── Salvar modelo .pkl ────────────────────────────────────────────────────────
# O modelo salvo será usado pelo Notebook 2 para calcular
# probabilidades de jogos futuros sem precisar retreinar

with open(PKL_FILE, "wb") as f:
    pickle.dump(modelos_salvos, f)
print(f"💾 {PKL_FILE} salvo ({len(modelos_salvos)} mercados)")

# ── Salvar base histórica final ───────────────────────────────────────────────
# Colunas finais limpas
cols_manter = (
    ["fixture_id","league","season_id","date_unix","home_id","away_id",
     "home_team","away_team","score_home","score_away","score_home_ht","score_away_ht",
     "odds_home","odds_draw","odds_away","odds_btts_y","odds_over25",
     "xg_home_pre","xg_away_pre","ppg_home","ppg_away",
     "elo_home","elo_away","elo_delta",
     "xg_n10_diff","xg_pre_diff","pct_vit_diff","ppg_diff","gols_marc_diff","gols_sof_diff"] +
    [c for c in df_feat.columns if f"_n{JANELA}_" in c] +
    [c for c in df_feat.columns if c.startswith("label_")] +
    [c for c in df_feat.columns if c.startswith("prob_")] +
    [c for c in df_feat.columns if c.startswith("ev_")]
)
cols_manter = [c for c in cols_manter if c in df_feat.columns]
df_out = df_feat[cols_manter].copy()
df_out.to_csv(BASE_CSV, index=False)

print(f"💾 {BASE_CSV} salvo!")
print(f"   {len(df_out):,} linhas × {df_out.shape[1]} colunas")
print(f"   Ligas: {df_out['league'].nunique()}")
print(f"\nColunas por tipo:")
print(f"  Features N{JANELA}: {len([c for c in df_out.columns if f'_n{JANELA}_' in c])}")
print(f"  ELO: {len([c for c in df_out.columns if 'elo' in c])}")
print(f"  Labels: {len([c for c in df_out.columns if c.startswith('label_')])}")
print(f"  Probabilidades: {len([c for c in df_out.columns if c.startswith('prob_')])}")
print(f"  EV: {len([c for c in df_out.columns if c.startswith('ev_')])}")


💾 modelo_v3.pkl salvo (6 mercados)
💾 base_historica_v3.csv salvo!
   6,027 linhas × 78 colunas
   Ligas: 31

Colunas por tipo:
  Features N10: 28
  ELO: 3
  Labels: 9
  Probabilidades: 6
  EV: 6


In [22]:
# ── Diagnóstico final — resumo de qualidade ──────────────────────────────────
print("="*65)
print("RESUMO DE QUALIDADE — Sistema v3")
print("="*65)

prob_cols = [(label, nome, odd, col_prob, col_ev) for label, nome, odd, col_prob, col_ev in MERCADOS
             if col_prob in df_out.columns]

for label, nome, odd, col_prob, col_ev in prob_cols:
    d = df_out[df_out[col_prob].notna()].copy()
    if len(d) == 0: continue
    be    = 1/odd
    mask_ev = d[col_ev] > EV_MIN
    n_ev    = mask_ev.sum()
    if n_ev == 0: continue
    acerto  = d.loc[mask_ev, label].mean()
    roi     = acerto * odd - 1
    ev_med  = d.loc[mask_ev, col_ev].mean()

    sinal = "✅" if roi > 0 else "❌"
    print(f"  {nome:<16} EV>{EV_MIN*100:.0f}%: {n_ev:>5} jogos | "
          f"acerto={acerto*100:.1f}% | ROI={roi*100:+.1f}% | EV_med={ev_med*100:+.1f}% {sinal}")

print()
print("📌 Próximo passo: rodar Notebook 2 (rodada_atual.ipynb)")
print("   → Carrega modelo_v3.pkl + elo_ratings.json")
print("   → Busca jogos futuros da API")
print("   → Calcula prob_base SEM usar odd")
print("   → Scheduler coleta odd 30min antes → calcula EV final")


RESUMO DE QUALIDADE — Sistema v3
  Back Casa        EV>3%:  1225 jogos | acerto=62.7% | ROI=+19.1% | EV_med=+19.4% ✅
  Back Fora        EV>3%:   977 jogos | acerto=49.3% | ROI=+23.3% | EV_med=+24.5% ✅
  BTTS             EV>3%:   562 jogos | acerto=55.7% | ROI=+3.0% | EV_med=+6.9% ✅
  Over 2.5         EV>3%:   102 jogos | acerto=60.8% | ROI=+9.4% | EV_med=+10.8% ✅
  Over 0.5 HT      EV>3%:   203 jogos | acerto=74.9% | ROI=+4.8% | EV_med=+6.7% ✅
  BTTS HT          EV>3%:     4 jogos | acerto=50.0% | ROI=+75.0% | EV_med=+7.1% ✅

📌 Próximo passo: rodar Notebook 2 (rodada_atual.ipynb)
   → Carrega modelo_v3.pkl + elo_ratings.json
   → Busca jogos futuros da API
   → Calcula prob_base SEM usar odd
   → Scheduler coleta odd 30min antes → calcula EV final
